In [132]:
import pandas as pd
import numpy as np
import re
from tableone import TableOne
from matplotlib import pyplot as plt

pd.set_option('display.max_rows', 100)  # Show up to 100 rows
pd.set_option('display.max_columns', None)

In [133]:
path = "/Users/herbert.kapesa/Library/CloudStorage/OneDrive-CentreforInfectiousDiseaseResearchinZambia/RMNCH/FIFSA/Data Extract/"


In [150]:
cols = pd.read_csv(path + 'FISFAZambia_DATA_2026-09-15_0529.csv', nrows=0).columns.tolist()
df = pd.read_csv(path + 'FISFAZambia_DATA_LABELS_2026-09-15_0540.csv', low_memory=False)

df.columns = cols

In [151]:
# Variables that will be spread across events
value_columns = [
    col for col in df.columns
    if col not in ["record_id", "redcap_event_name"]
]
# transform the dataset from long to wide format
    # Convert long -> wide
df = df.pivot(
    index="record_id",
    columns="redcap_event_name",
    values=value_columns
)
# Flatten the MultiIndex columns
df.columns = [
    f"{variable}_{event}"
    for variable, event in df.columns
]
# Convert record_id back into a normal column
df = df.reset_index()


In [ ]:
df = df.dropna()

In [ ]:
df = df.dropna(axis=1, how="all")

In [152]:
df


redcap_survey_identifier_1 Month redcap_survey_identifier_3 Month  \
record_id                                                                     
1                                      NaN                              NaN   
2                                      NaN                              NaN   
3                                      NaN                              NaN   
4                                      NaN                              NaN   
5                                      NaN                              NaN   
...                                    ...                              ...   
246                                    NaN                              NaN   
247                                    NaN                              NaN   
248                                    NaN                              NaN   
249                                    NaN                              NaN   
250                                    NaN                              NaN   

          redcap_survey_identifier_6 Month  \
record_id                                    
1                                      NaN   
2                                      NaN   
3                                      NaN   
4                                      NaN   
5                                      NaN   
...                                    ...   
246                                    NaN   
247                                    NaN   
248                                    NaN   
249                                    NaN   
250                                    NaN   

          redcap_survey_identifier_Enrollment introduction_timestamp_1 Month  \
record_id                                                                      
1                                         NaN                            NaN   
2                                         NaN                            NaN   
3                                         NaN                            NaN   
4                                         NaN                            NaN   
5                                         NaN                            NaN   
...                                       ...                            ...   
246                                       NaN                            NaN   
247                                       NaN                            NaN   
248                                       NaN                            NaN   
249                                       NaN                            NaN   
250                                       NaN                            NaN   

          introduction_timestamp_3 Month introduction_timestamp_6 Month  \
record_id                                                                 
1                                    NaN                            NaN   
2                                    NaN                            NaN   
3                                    NaN                            NaN   
4                                    NaN                            NaN   
5                                    NaN                            NaN   
...                                  ...                            ...   
246                                  NaN                            NaN   
247                                  NaN                            NaN   
248                                  NaN                            NaN   
249                                  NaN                            NaN   
250                                  NaN                            NaN   

          introduction_timestamp_Enrollment living_situation_1 Month  \
record_id                                                              
1                                       NaN                      NaN   
2                                       NaN                      NaN   
3                       2026-04-10 04:21:05                      NaN   
4     

In [139]:
print([f"{variable}_{event}"
       for variable, event in df.columns])

['redcap_survey_identifier_1 Month', 'redcap_survey_identifier_3 Month', 'redcap_survey_identifier_6 Month', 'redcap_survey_identifier_Enrollment', 'introduction_timestamp_1 Month', 'introduction_timestamp_3 Month', 'introduction_timestamp_6 Month', 'introduction_timestamp_Enrollment', 'living_situation_1 Month', 'living_situation_3 Month', 'living_situation_6 Month', 'living_situation_Enrollment', 'living_roommate_1 Month', 'living_roommate_3 Month', 'living_roommate_6 Month', 'living_roommate_Enrollment', 'living_coenrolled_id_1 Month', 'living_coenrolled_id_3 Month', 'living_coenrolled_id_6 Month', 'living_coenrolled_id_Enrollment', 'living_other_1 Month', 'living_other_3 Month', 'living_other_6 Month', 'living_other_Enrollment', 'intro_notes_1 Month', 'intro_notes_3 Month', 'intro_notes_6 Month', 'intro_notes_Enrollment', 'par_joindate_1 Month', 'par_joindate_3 Month', 'par_joindate_6 Month', 'par_joindate_Enrollment', 'par_joindate_utc_1 Month', 'par_joindate_utc_3 Month', 'par_jo

## Data cleaning and preparation

In [87]:
columns =[
    "age_a",
    "race",
    "weight_a",
    "height_a",
    "blood_pressure_s_a",
    "blood_pressure_d_a",
    "blood_pressure_summ_a",
    "education_a",
    "income_a",
    "employed_a",
    "occupation"
]
test_cols = [
    "hemog_Enrollment",
    "serum_folate_e_Enrollment",
    "rbc_folate_e_Enrollment",
    "hemog_1 Month",
    "serum_folate_1_Enrollment",
    "rbc_folate_1_Enrollment",
    "hemog_3 Month",
    "serum_folate_3_Enrollment",
    "rbc_folate_3_Enrollment"
]



In [120]:
df['rbc_folate_e_Enrollment'].value_counts()

rbc_folate_e_Enrollment
297.0    4
266.0    3
209.0    3
221.0    3
347.0    3
        ..
595.0    1
709.0    1
312.0    1
511.0    1
282.0    1
Name: count, Length: 185, dtype: int64

# Hemoglobin 

In [95]:

import plotly.graph_objects as go

hb_cols = [
    "hemog_Enrollment",
    "hemog_1 Month",
    "hemog_3 Month"
]

hb_cutoff = 12.0

df_alluvial = df.copy()

# Create categories while preserving missing values
for col in hb_cols:

    category_col = col + "_category"

    # Start with genuine missing values
    df_alluvial[category_col] = pd.Series(
        pd.NA,
        index=df_alluvial.index,
        dtype="string"
    )

    # Low Hb
    df_alluvial.loc[
        df_alluvial[col].notna() & (df_alluvial[col] < hb_cutoff),
        category_col
    ] = "Low"

    # Normal Hb
    df_alluvial.loc[
        df_alluvial[col].notna() & (df_alluvial[col] >= hb_cutoff),
        category_col
    ] = "Normal"

In [96]:
category_cols = [
    "hemog_Enrollment_category",
    "hemog_1 Month_category",
    "hemog_3 Month_category"
]

plot_df = (
    df_alluvial[
        ["record_id"] + category_cols
    ]
    .dropna(subset=category_cols)
    .copy()
)

print("Participants included:", len(plot_df))

display(plot_df.head())

Participants included: 121


,record_id,hemog_Enrollment_category,hemog_1 Month_category,hemog_3 Month_category
0,1,Normal,Normal,Normal
1,2,Normal,Low,Normal
2,3,Normal,Normal,Normal
3,4,Normal,Normal,Normal
4,5,Normal,Normal,Low


In [97]:
for col in category_cols:
    print(col)
    print(plot_df[col].value_counts(dropna=False))
    print()

hemog_Enrollment_category
hemog_Enrollment_category
Normal    91
Low       30
Name: count, dtype: Int64

hemog_1 Month_category
hemog_1 Month_category
Normal    83
Low       38
Name: count, dtype: Int64

hemog_3 Month_category
hemog_3 Month_category
Normal    89
Low       32
Name: count, dtype: Int64



In [98]:
# Enrollment -> 1 Month
flow_1 = (
    plot_df
    .groupby(
        [
            "hemog_Enrollment_category",
            "hemog_1 Month_category"
        ],
        observed=True
    )
    .size()
    .reset_index(name="count")
)

# 1 Month -> 3 Months
flow_2 = (
    plot_df
    .groupby(
        [
            "hemog_1 Month_category",
            "hemog_3 Month_category"
        ],
        observed=True
    )
    .size()
    .reset_index(name="count")
)


# Define nodes
nodes = [
    "Enrollment: Low",
    "Enrollment: Normal",
    "1 Month: Low",
    "1 Month: Normal",
    "3 Months: Low",
    "3 Months: Normal"
]

node_index = {
    node: i for i, node in enumerate(nodes)
}


sources = []
targets = []
values = []


# Enrollment -> 1 Month
for _, row in flow_1.iterrows():

    source = f"Enrollment: {row['hemog_Enrollment_category']}"
    target = f"1 Month: {row['hemog_1 Month_category']}"

    sources.append(node_index[source])
    targets.append(node_index[target])
    values.append(row["count"])


# 1 Month -> 3 Months
for _, row in flow_2.iterrows():

    source = f"1 Month: {row['hemog_1 Month_category']}"
    target = f"3 Months: {row['hemog_3 Month_category']}"

    sources.append(node_index[source])
    targets.append(node_index[target])
    values.append(row["count"])


# Plot
fig = go.Figure(
    go.Sankey(
        arrangement="snap",

        node=dict(
            pad=20,
            thickness=25,
            label=nodes
        ),

        link=dict(
            source=sources,
            target=targets,
            value=values
        )
    )
)

fig.update_layout(
    title="Haemoglobin Status Across Enrollment, 1 Month and 3 Months",
    font_size=13,
    width=1000,
    height=600
)

fig.show()

# Folate

In [114]:
df["serum_folate_e_Enrollment"].loc[df["serum_folate_e_Enrollment"] > 20]

15     22.6
23     33.6
24     45.3
32     31.8
44     20.3
50     22.8
61     21.4
74     20.8
93     20.3
96     22.5
98     24.5
100    26.1
104    21.5
107    23.6
116    29.6
117    22.4
118    20.8
124    20.9
125    23.3
128    20.9
133    23.3
137    45.3
145    20.4
148    21.2
154    20.1
180    21.2
213    40.6
216    24.6
218    21.0
226    25.8
235    45.3
236    20.3
242    29.7
244    25.4
Name: serum_folate_e_Enrollment, dtype: object

In [126]:

# -----------------------------------
# 1. Define folate variables
# -----------------------------------

folate_cols = [
    "serum_folate_e_Enrollment",
    "serum_folate_1_Enrollment",
    #"serum_folate_3_Enrollment"
]

# Set this to the cutoff used in your study
folate_cutoff = 25.5


# -----------------------------------
# 2. Categorise folate as Low/Normal
# -----------------------------------

df_folate = df.copy()

for col in folate_cols:

    category_col = col + "_category"

    # Create a proper nullable string column
    df_folate[category_col] = pd.Series(
        pd.NA,
        index=df_folate.index,
        dtype="string"
    )

    # Low folate
    df_folate.loc[
        df_folate[col].notna() &
        (df_folate[col] < folate_cutoff),
        category_col
    ] = "Low"

    # Normal folate
    df_folate.loc[
        df_folate[col].notna() &
        (df_folate[col] >= folate_cutoff),
        category_col
    ] = "Normal"

In [127]:
category_cols = [
    "serum_folate_e_Enrollment_category",
    "serum_folate_1_Enrollment_category",
    #"serum_folate_3_Enrollment_category"
]

plot_folate = (
    df_folate[
        ["record_id"] + category_cols
    ]
    .dropna(subset=category_cols)
    .copy()
)

print("Participants included:", len(plot_folate))

display(plot_folate.head())

Participants included: 177


,record_id,serum_folate_e_Enrollment_category,serum_folate_1_Enrollment_category
0,1,Low,Low
1,2,Low,Low
2,3,Low,Low
3,4,Low,Low
4,5,Low,Low


In [128]:
for col in category_cols:
    print(col)
    print(plot_folate[col].value_counts(dropna=False))
    print()

serum_folate_e_Enrollment_category
serum_folate_e_Enrollment_category
Low       169
Normal      8
Name: count, dtype: Int64

serum_folate_1_Enrollment_category
serum_folate_1_Enrollment_category
Low       156
Normal     21
Name: count, dtype: Int64



In [131]:
# -----------------------------------
# 3. Calculate transitions
# -----------------------------------

# Enrollment -> 1 Month
flow_1 = (
    plot_folate
    .groupby(
        [
            "serum_folate_e_Enrollment_category",
            "serum_folate_1_Enrollment_category"
        ],
        observed=True
    )
    .size()
    .reset_index(name="count")
)

# # 1 Month -> 3 Months
# flow_2 = (
#     plot_folate
#     .groupby(
#         [
#             "serum_folate_1_Enrollment_category",
#             "serum_folate_3_Enrollment_category"
#         ],
#         observed=True
#     )
#     .size()
#     .reset_index(name="count")
# )


# -----------------------------------
# 4. Define nodes
# -----------------------------------

nodes = [
    "Enrollment: Low",
    "Enrollment: Normal",
    "1 Month: Low",
    "1 Month: Normal",
    # "3 Months: Low",
    # "3 Months: Normal"
]

node_index = {
    node: i for i, node in enumerate(nodes)
}


# -----------------------------------
# 5. Build source-target flows
# -----------------------------------

sources = []
targets = []
values = []


# Enrollment -> 1 Month
for _, row in flow_1.iterrows():

    source = (
        f"Enrollment: "
        f"{row['serum_folate_e_Enrollment_category']}"
    )

    target = (
        f"1 Month: "
        f"{row['serum_folate_1_Enrollment_category']}"
    )

    sources.append(node_index[source])
    targets.append(node_index[target])
    values.append(row["count"])


# # 1 Month -> 3 Months
# for _, row in flow_2.iterrows():

#     source = (
#         f"1 Month: "
#         f"{row['serum_folate_1_Enrollment_category']}"
#     )

#     # target = (
#     #     f"3 Months: "
#     #     f"{row['serum_folate_3_Enrollment_category']}"
#     # )

#     sources.append(node_index[source])
#     targets.append(node_index[target])
#     values.append(row["count"])


# -----------------------------------
# 6. Plot Sankey / Alluvial diagram
# -----------------------------------

fig = go.Figure(
    go.Sankey(
        arrangement="snap",

        node=dict(
            pad=20,
            thickness=25,
            label=nodes
        ),

        link=dict(
            source=sources,
            target=targets,
            value=values
        )
    )
)

fig.update_layout(
    title="Serum Folate Status Across Enrollment, 1 Month and 3 Months",
    font_size=13,
    width=1000,
    height=600
)

fig.show()

# RBC Folate

In [123]:
import pandas as pd
import plotly.graph_objects as go

# -----------------------------------
# 1. Define RBC folate variables
# -----------------------------------

rbc_folate_cols = [
    "rbc_folate_e_Enrollment",
    "rbc_folate_1_Enrollment",
    "rbc_folate_3_Enrollment"
]

rbc_folate_cutoff = 400


# -----------------------------------
# 2. Categorise as Low / Normal
# -----------------------------------

df_rbc = df.copy()

for col in rbc_folate_cols:

    category_col = col + "_category"

    # Create proper nullable category column
    df_rbc[category_col] = pd.Series(
        pd.NA,
        index=df_rbc.index,
        dtype="string"
    )

    # Low
    df_rbc.loc[
        df_rbc[col].notna() &
        (df_rbc[col] < rbc_folate_cutoff),
        category_col
    ] = "Low"

    # Normal
    df_rbc.loc[
        df_rbc[col].notna() &
        (df_rbc[col] >= rbc_folate_cutoff),
        category_col
    ] = "Normal"

In [124]:
category_cols = [
    "rbc_folate_e_Enrollment_category",
    "rbc_folate_1_Enrollment_category",
    "rbc_folate_3_Enrollment_category"
]

plot_rbc = (
    df_rbc[
        ["record_id"] + category_cols
    ]
    .dropna(subset=category_cols)
    .copy()
)

print("Participants included:", len(plot_rbc))

display(plot_rbc.head())

for col in category_cols:
    print(col)
    print(plot_rbc[col].value_counts(dropna=False))
    print()

Participants included: 58


,record_id,rbc_folate_e_Enrollment_category,rbc_folate_1_Enrollment_category,rbc_folate_3_Enrollment_category
0,1,Low,Low,Low
1,2,Low,Low,Low
2,3,Low,Low,Low
3,4,Low,Low,Low
4,5,Low,Low,Normal


rbc_folate_e_Enrollment_category
rbc_folate_e_Enrollment_category
Low       47
Normal    11
Name: count, dtype: Int64

rbc_folate_1_Enrollment_category
rbc_folate_1_Enrollment_category
Low       46
Normal    12
Name: count, dtype: Int64

rbc_folate_3_Enrollment_category
rbc_folate_3_Enrollment_category
Low       38
Normal    20
Name: count, dtype: Int64



In [125]:
# -----------------------------------
# 3. Calculate transitions
# -----------------------------------

# Enrollment -> 1 Month
flow_1 = (
    plot_rbc
    .groupby(
        [
            "rbc_folate_e_Enrollment_category",
            "rbc_folate_1_Enrollment_category"
        ],
        observed=True
    )
    .size()
    .reset_index(name="count")
)

# 1 Month -> 3 Months
flow_2 = (
    plot_rbc
    .groupby(
        [
            "rbc_folate_1_Enrollment_category",
            "rbc_folate_3_Enrollment_category"
        ],
        observed=True
    )
    .size()
    .reset_index(name="count")
)


# -----------------------------------
# 4. Define nodes
# -----------------------------------

nodes = [
    "Enrollment: Low",
    "Enrollment: Normal",
    "1 Month: Low",
    "1 Month: Normal",
    "3 Months: Low",
    "3 Months: Normal"
]

node_index = {
    node: i for i, node in enumerate(nodes)
}


# -----------------------------------
# 5. Build source-target flows
# -----------------------------------

sources = []
targets = []
values = []


# Enrollment -> 1 Month
for _, row in flow_1.iterrows():

    source = (
        f"Enrollment: "
        f"{row['rbc_folate_e_Enrollment_category']}"
    )

    target = (
        f"1 Month: "
        f"{row['rbc_folate_1_Enrollment_category']}"
    )

    sources.append(node_index[source])
    targets.append(node_index[target])
    values.append(row["count"])


# 1 Month -> 3 Months
for _, row in flow_2.iterrows():

    source = (
        f"1 Month: "
        f"{row['rbc_folate_1_Enrollment_category']}"
    )

    target = (
        f"3 Months: "
        f"{row['rbc_folate_3_Enrollment_category']}"
    )

    sources.append(node_index[source])
    targets.append(node_index[target])
    values.append(row["count"])


# -----------------------------------
# 6. Plot Sankey / Alluvial diagram
# -----------------------------------

fig = go.Figure(
    go.Sankey(
        arrangement="snap",

        node=dict(
            pad=20,
            thickness=25,
            label=nodes
        ),

        link=dict(
            source=sources,
            target=targets,
            value=values
        )
    )
)

fig.update_layout(
    title="Red Blood Cell Folate Status Across Enrollment, 1 Month and 3 Months",
    font_size=13,
    width=1000,
    height=600
)

fig.show()